# 06 -- Train / Validation / Test Split
**Purpose:** Partition the dataset into train (70%), validation (20%), test (10%) using temporal split.

**Input:** `data/processed/validated_features.parquet`
**Output:** X_train, X_val, X_test, y_train, y_val, y_test (parquet files in data/processed/)

## Learning Objectives
- Understand why temporal split is mandatory for fraud detection
- Visualise what data leakage from the future looks like in a random split
- Verify class balance is preserved in each fold
- Understand the role of validation set vs test set

## Business Context
Fraud patterns evolve. A model trained on months 1-9 must detect fraud from months 10-12,
including new typologies that emerged after September. A random split mixes future signals
into training, making every metric look better than it will be in production.

## Step 1 -- Before You Split
Answer in a markdown cell:
1. Why can you NOT use train_test_split(shuffle=True) for this dataset?
2. If you randomly split, which feature category would be most affected by leakage? Why?
3. What is the difference between validation set and test set? What is each used for?
4. Why do you need to sort by timestamp BEFORE splitting?

## Step 2 -- Sort by Timestamp
Sort the feature matrix by transaction date.
Print the earliest and latest date. Confirm data spans the expected range.

## Step 3 -- Apply Temporal Split
Apply 70 / 20 / 10 split. Print shape of each partition.
Print fraud rate in each partition -- it should be approximately 0.49% in all three.
If it varies significantly, investigate why before proceeding.

## Step 4 -- Verify No Leakage
Confirm zero date overlap between splits:
- max(train dates) < min(val dates)
- max(val dates) < min(test dates)
Print these boundary dates.

## Step 5 -- Save All Partitions
Save all 6 arrays to data/processed/. Confirm file sizes on disk.

## Definition of Done
- [ ] Temporal sort verified
- [ ] Zero date overlap between any split
- [ ] Fraud rate approximately equal in train/val/test
- [ ] All 6 partition files saved
- [ ] You can explain in one paragraph why temporal split is non-negotiable for fraud

In [2]:
import pandas as pd

X_clean = pd.read_parquet(r"../data/processed\feature_matrix_clean.parquet")
y = pd.read_parquet(r"../data/processed\target.parquet").squeeze()

print(X_clean.shape, y.value_counts().to_dict())


(50800, 46) {0: 50550, 1: 250}


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train: {X_train.shape} | Fraud: {y_train.sum()} ({y_train.mean():.3%})")
print(f"Test:  {X_test.shape}  | Fraud: {y_test.sum()} ({y_test.mean():.3%})")


Train: (40640, 46) | Fraud: 200 (0.492%)
Test:  (10160, 46)  | Fraud: 50 (0.492%)


In [4]:
import numpy as np

X_train.to_parquet(r"../data/processed\X_train.parquet", index=False)
X_test.to_parquet(r"../data/processed\X_test.parquet", index=False)
y_train.to_frame().to_parquet(r"../data/processed\y_train.parquet", index=False)
y_test.to_frame().to_parquet(r"../data/processed\y_test.parquet", index=False)

print("Saved: X_train, X_test, y_train, y_test")


Saved: X_train, X_test, y_train, y_test
